Assignment -2
Dear All: 
As discussed in the class, please sentiment prediction task on the attached tweet dataset.
Steps:
1. Preprocessing (like I said)
    1.1: Tokenization
    1.2: Remove special characters, stop-words
    1.3: get the modified tweets
    1.4: list out unique words through out the document
    1.5 : Calculate TF*IDF values for each of the unique words
2. Use TF*IDF as the feature
3. Build features matrix file (X file) for training and test set
4. Construct the file with labels (Y file)
5. feed X and Y file into an ML model that you know
6. Train the model
7. Test the model bu feeding the X file for Test dataset
8. get the prediction
9. Compare it with the labels in the test dataset.


In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
def parse_tweet_file(filepath):
    tweets = []
    labels = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Split by commas, then group every 10 fields
            parts = line.split(',')
            # Process chunks of 10
            for i in range(0, len(parts) - 9, 10):
                chunk = parts[i:i+10]
                if len(chunk) == 10:
                    # Field index 1 = full tweet, index 3 = sentiment
                    tweet_text = chunk[1].strip()
                    sentiment = chunk[3].strip().lower()
                    if sentiment in ['positive', 'negative', 'neutral']:
                        tweets.append(tweet_text)
                        labels.append(sentiment)
    return tweets, labels

# Load data
train_tweets, train_labels = parse_tweet_file('train.csv')
test_tweets, test_labels = parse_tweet_file('test.csv')

print(f"Train: {len(train_tweets)} samples")
print(f"Test:  {len(test_tweets)} samples")

Train: 20971 samples
Test:  627 samples


In [3]:
def clean_tweet(tweet):
    if not isinstance(tweet, str):
        return ""
    # Lowercase
    tweet = tweet.lower()
    # Remove URLs, mentions, hashtags
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet, flags=re.MULTILINE)
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    # Keep only letters and spaces
    tweet = re.sub(r'[^a-z\s]', ' ', tweet)
    # Remove extra spaces and short words
    words = [w for w in tweet.split() if len(w) > 2]
    return ' '.join(words)

# Apply cleaning
train_clean = [clean_tweet(t) for t in train_tweets]
test_clean = [clean_tweet(t) for t in test_tweets]

In [4]:
# Vectorize (limit features to avoid memory issues)
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),      # unigrams + bigrams
    stop_words='english',    # built-in stopword list
    min_df=2,                # ignore very rare words
    max_df=0.95              # ignore very common words
)

X_train = vectorizer.fit_transform(train_clean)
X_test = vectorizer.transform(test_clean)

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(train_labels)
y_test = le.transform(test_labels)

# Train model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [5]:
y_pred = model.predict(X_test)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))

=== Classification Report ===
              precision    recall  f1-score   support

    negative       0.61      0.36      0.46       179
     neutral       0.46      0.74      0.57       238
    positive       0.72      0.48      0.58       210

    accuracy                           0.55       627
   macro avg       0.60      0.53      0.54       627
weighted avg       0.59      0.55      0.54       627



In [6]:
# Load manual test set (classic 2009)
manual = pd.read_csv(
    'testdata.manual.2009.06.14.csv',
    header=None,
    names=['polarity', 'id', 'date', 'query', 'user', 'text'],
    encoding='latin1'
)

# Map polarity: 0=neg, 2=neutral, 4=pos
polarity_map = {0: 'negative', 2: 'neutral', 4: 'positive'}
manual = manual[manual['polarity'].isin([0, 2, 4])]
manual['sentiment'] = manual['polarity'].map(polarity_map)

# Clean and predict
manual_clean = [clean_tweet(t) for t in manual['text']]
X_manual = vectorizer.transform(manual_clean)
y_manual = le.transform(manual['sentiment'])

y_manual_pred = model.predict(X_manual)
print("\n=== Manual Test Set (2009) ===")
print(classification_report(y_manual, y_manual_pred, target_names=le.classes_))


=== Manual Test Set (2009) ===
              precision    recall  f1-score   support

    negative       0.86      0.44      0.59       178
     neutral       0.45      0.93      0.61       140
    positive       0.83      0.58      0.68       198

    accuracy                           0.63       516
   macro avg       0.71      0.65      0.62       516
weighted avg       0.74      0.63      0.63       516



In [ ]:
# -----------------------------
# LIVE SENTIMENT PREDICTION
# -----------------------------

def predict_live_sentiment(tweet_text):
    """
    Clean and predict sentiment for a single tweet using your trained model.
    """
    cleaned = clean_tweet(tweet_text)
    X_new = vectorizer.transform([cleaned])
    pred_label = model.predict(X_new)[0]
    sentiment = le.inverse_transform([pred_label])[0]
    return sentiment

# Allow live input in Jupyter Notebook
print("🔮 Enter a tweet below to analyze its sentiment (positive / neutral / negative):")
while True:
    user_tweet = input("\nEnter a tweet (or type 'quit' to stop): ")
    if user_tweet.lower() in ['quit', 'exit', 'stop']:
        print("👋 Goodbye!")
        break
    if user_tweet.strip() == "":
        print("⚠️ Empty tweet. Try again.")
        continue
    result = predict_live_sentiment(user_tweet)
    print(f"✅ Sentiment: {result}")

🔮 Enter a tweet below to analyze its sentiment (positive / neutral / negative):



Enter a tweet (or type 'quit' to stop):  the world should burn in hell


✅ Sentiment: negative
